In [63]:
# CELL 1: MODEL DEFINITION AND LOADING

import os
import warnings
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import (
    Conv2D, Dense, Dropout, Flatten, GlobalAveragePooling2D, Input, MaxPooling2D
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

GROUP_ID = "s3715228_s3343711_s4139514"
MODEL_H5 = f"model_{GROUP_ID}.h5"
MODEL_KERAS = f"model_{GROUP_ID}.keras"

print("--- Environment ---")
print(f"TensorFlow: {tf.__version__} | Keras: {keras.__version__}")
print(f"Group ID: {GROUP_ID}")

# MODEL DEFINITION
def build_mtl_model(input_shape=(32, 32, 1)):
    inputs = Input(shape=input_shape)

    x = Conv2D(32, 3, padding="same", activation="relu")(inputs)
    x = MaxPooling2D(2)(x)
    x = Conv2D(64, 3, padding="same", activation="relu")(x)
    x = MaxPooling2D(2)(x)

    a = Conv2D(64, 3, padding="same", activation="relu")(x)
    a = MaxPooling2D(2)(a)
    a = Flatten()(a)
    a = Dense(128, activation="relu")(a)
    a = Dropout(0.6)(a)
    output_A = Dense(10, activation="softmax", name="output_A")(a)

    b = GlobalAveragePooling2D(name="B_gap")(x)
    b = Dense(64, activation="relu", name="B_dense")(b)
    b = Dropout(0.6, name="B_dropout")(b)
    output_B = Dense(32, activation="softmax", name="output_B")(b)

    c = GlobalAveragePooling2D()(x)
    c = Dense(64, activation="relu")(c)
    output_C = Dense(1, activation="linear", name="output_C")(c)

    model = Model(
        inputs=inputs,
        outputs={"output_A": output_A, "output_B": output_B, "output_C": output_C},
        name="MTL_Model"
    )

    model.compile(
        optimizer=Adam(learning_rate=3e-4),
        loss={
            "output_A": "sparse_categorical_crossentropy",
            "output_B": "sparse_categorical_crossentropy",
            "output_C": "mse",
        },
        loss_weights={"output_A": 1.2, "output_B": 0.7, "output_C": 0.4},
        metrics={"output_A": "accuracy", "output_B": "accuracy", "output_C": "mae"},
    )
    return model

# BUILD AND LOAD WEIGHTS
print("\n--- Model ---")
model = build_mtl_model()
print(f"Architecture: MTL_Model ({model.count_params():,} parameters)")
print(f"  - Output A: 10-class classification (softmax)")
print(f"  - Output B: 32-class classification (softmax)")
print(f"  - Output C: Regression (linear)")

h5_exists = os.path.exists(MODEL_H5)
keras_exists = os.path.exists(MODEL_KERAS)

print("\n--- Weights ---")
print(f"{MODEL_H5}: {'Found' if h5_exists else 'Not found'}")
print(f"{MODEL_KERAS}: {'Found' if keras_exists else 'Not found'}")

if h5_exists or keras_exists:
    weights_file = MODEL_H5 if h5_exists else MODEL_KERAS
    model.load_weights(weights_file)
    print(f"Loaded weights from: {weights_file}")
else:
    print("\nNo weights found. Training new model...")
    data = np.load('dataset_dev_3000.npz')
    X, y = data['X'], data['y']
    
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y[:, 0])
    
    X_train_mtl = X_train[..., None].astype('float32')
    X_val_mtl = X_val[..., None].astype('float32')
    mean, std = X_train_mtl.mean(), X_train_mtl.std() + 1e-6
    X_train_mtl, X_val_mtl = (X_train_mtl - mean) / std, (X_val_mtl - mean) / std
    
    y_A_train, y_B_train, y_C_train = y_train[:, 0], y_train[:, 1], y_train[:, 2]
    y_A_val, y_B_val, y_C_val = y_val[:, 0], y_val[:, 1], y_val[:, 2]
    
    classes = np.unique(y_B_train)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_B_train)
    class_weight_B = dict(zip(classes, weights))
    sample_weight = {
        "output_A": np.ones(len(y_A_train), dtype=np.float32),
        "output_B": np.array([class_weight_B[y] for y in y_B_train], dtype=np.float32),
        "output_C": np.ones(len(y_C_train), dtype=np.float32),
    }
    
    callbacks = [
        EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=6, min_lr=1e-5),
    ]
    
    print(f"Training on {len(X_train)} samples, validating on {len(X_val)} samples...")
    model.fit(
        X_train_mtl,
        {"output_A": y_A_train, "output_B": y_B_train, "output_C": y_C_train},
        sample_weight=sample_weight,
        validation_data=(X_val_mtl, {"output_A": y_A_val, "output_B": y_B_val, "output_C": y_C_val}),
        epochs=80, batch_size=64, callbacks=callbacks, verbose=2,
    )
    
    model.save(MODEL_H5)
    model.save(MODEL_KERAS)
    print(f"\nSaved: {MODEL_H5}, {MODEL_KERAS}")

print("\nModel ready for inference.")

--- Environment ---
TensorFlow: 2.20.0 | Keras: 3.12.0
Group ID: s3715228_s3343711_s4139514

--- Model ---
Architecture: MTL_Model (198,699 parameters)
  - Output A: 10-class classification (softmax)
  - Output B: 32-class classification (softmax)
  - Output C: Regression (linear)

--- Weights ---
model_s3715228_s3343711_s4139514.h5: Not found
model_s3715228_s3343711_s4139514.keras: Not found

No weights found. Training new model...
Training on 2400 samples, validating on 600 samples...
Epoch 1/80
38/38 - 2s - 64ms/step - loss: 5.2449 - output_A_accuracy: 0.1171 - output_A_loss: 2.3052 - output_B_accuracy: 0.0283 - output_B_loss: 3.4757 - output_C_loss: 0.1121 - output_C_mae: 0.2798 - val_loss: 5.1863 - val_output_A_accuracy: 0.1250 - val_output_A_loss: 2.2760 - val_output_B_accuracy: 0.0317 - val_output_B_loss: 3.4632 - val_output_C_loss: 0.0776 - val_output_C_mae: 0.2389 - learning_rate: 3.0000e-04
Epoch 2/80
38/38 - 1s - 22ms/step - loss: 5.1852 - output_A_accuracy: 0.1408 - output_


Saved: model_s3715228_s3343711_s4139514.h5, model_s3715228_s3343711_s4139514.keras

Model ready for inference.


In [64]:
# CELL 2: DATA PROCESSING AND PREDICTION

# Load dataset
data = np.load('dataset_dev_3000.npz')
X = data['X']  # (N, 32, 32)
y = data['y']  # (N, 3)

print("--- Data ---")
print(f"Input: {X.shape} | Targets: {y.shape}")

# Preprocessing (same as training)
X_processed = X[..., None].astype('float32')
mean, std = X_processed.mean(), X_processed.std() + 1e-6
X_processed = (X_processed - mean) / std
print(f"Preprocessed: {X_processed.shape} (normalized: mean={mean:.4f}, std={std:.4f})")

# Prediction
print("\n--- Prediction ---")
predictions = model.predict(X_processed, verbose=0)

pred_A_labels = np.argmax(predictions["output_A"], axis=1)
pred_B_labels = np.argmax(predictions["output_B"], axis=1)
pred_C_values = predictions["output_C"].squeeze()

print(f"Generated predictions for {len(X)} samples")

# Evaluation
y_A, y_B, y_C = y[:, 0], y[:, 1], y[:, 2]

acc_A = np.mean(pred_A_labels == y_A)
acc_B = np.mean(pred_B_labels == y_B)
mae_C = np.mean(np.abs(pred_C_values - y_C))

print("\n--- Results ---")
print(f"Target A (10-class): {acc_A*100:.2f}% accuracy (random baseline: 10.00%)")
print(f"Target B (32-class): {acc_B*100:.2f}% accuracy (random baseline: 3.12%)")
print(f"Target C (regression): MAE = {mae_C:.4f} (range: [{y_C.min():.2f}, {y_C.max():.2f}])")

--- Data ---
Input: (3000, 32, 32) | Targets: (3000, 3)
Preprocessed: (3000, 32, 32, 1) (normalized: mean=0.8141, std=0.7387)

--- Prediction ---
Generated predictions for 3000 samples

--- Results ---
Target A (10-class): 57.30% accuracy (random baseline: 10.00%)
Target B (32-class): 4.30% accuracy (random baseline: 3.12%)
Target C (regression): MAE = 0.1389 (range: [0.00, 1.00])
